In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.utils import resample
from collections import defaultdict

In [2]:
# TCVファイルの読み込み
train_data = pd.read_csv('../000.data/train/train_C.tsv', sep="\t")
test = pd.read_csv('../000.data/test.tsv', sep="\t")

In [3]:
# 末尾が "_C" のテストデータだけを抽出
test_data = test[test["user_id"].str.endswith("_C")]

In [4]:
# 関連度スコアの設定
def compute_relevance(row):
    if row["event_type"] == 3 and row["ad"] == 1:
        return 3  # 広告経由の購入
    elif row["event_type"] == 2:
        return 2  # 広告クリック
    elif row["event_type"] == 1:
        return 1  # 詳細ページ閲覧
    else:
        return 0  # カート追加

train_data["relevance"] = train_data.apply(compute_relevance, axis=1)

In [5]:
# タイムスタンプ処理
train_data["time_stamp"] = pd.to_datetime(train_data["time_stamp"])
train_data["time_since_last"] = (train_data["time_stamp"].max() - train_data["time_stamp"]).dt.total_seconds()

In [6]:
# ユーザーと商品の特徴量
user_features = train_data.groupby("user_id").agg(
    user_total_interactions=("product_id", "count"),
    user_unique_products=("product_id", "nunique"),
    user_avg_time_since=("time_since_last", "mean")
).reset_index()

product_features = train_data.groupby("product_id").agg(
    product_total_interactions=("user_id", "count"),
    product_unique_users=("user_id", "nunique")
).reset_index()

In [7]:
# **新しい特徴量（ユーザー・商品間の関係性）**
interaction_features = train_data.groupby(["user_id", "product_id"]).agg(
    user_product_interaction_count=("event_type", "count"),
    user_product_avg_time_since=("time_since_last", "mean")
).reset_index()

In [8]:
# データ結合
train_data = train_data.merge(user_features, on="user_id", how="left")
train_data = train_data.merge(product_features, on="product_id", how="left")
train_data = train_data.merge(interaction_features, on=["user_id", "product_id"], how="left")

In [9]:
# 学習用特徴量
features = [
    "user_total_interactions", "user_unique_products", "user_avg_time_since",
    "product_total_interactions", "product_unique_users",
    "user_product_interaction_count", "user_product_avg_time_since"
]

In [10]:
# データバランス調整（リサンプリング最適化）
train_data_pos = train_data[train_data["relevance"] >= 2]  
train_data_neg = train_data[train_data["relevance"] < 2]   

train_data_neg_undersampled = resample(train_data_neg, replace=True, n_samples=700000, random_state=42)
train_data_pos_oversampled = resample(train_data_pos, replace=True, n_samples=800000, random_state=42)

train_data_balanced = pd.concat([train_data_neg_undersampled, train_data_pos_oversampled])

In [11]:
# クロスバリデーション
kf = KFold(n_splits=5, shuffle=True, random_state=42)
ndcg_scores = []

In [12]:
for train_index, val_index in kf.split(train_data_balanced):
    train_set = train_data_balanced.iloc[train_index]
    val_set = train_data_balanced.iloc[val_index]

    X_train, y_train = train_set[features], train_set["relevance"]
    X_val, y_val = val_set[features], val_set["relevance"]

    group_train = train_set.groupby("user_id").size().to_numpy()
    group_val = val_set.groupby("user_id").size().to_numpy()

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtrain.set_group(group_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    dval.set_group(group_val)

    params = {
        "objective": "rank:ndcg",
        "eval_metric": "ndcg",
        "booster": "gbtree",
        "eta": 0.05,  # 学習率を下げて汎化性能向上
        "max_depth": 7,
        "min_child_weight": 20,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "lambda": 1.5,
        "gamma": 0.2
    }

    evals_result = {}
    model = xgb.train(params, dtrain, num_boost_round=500,
                      evals=[(dtrain, "train"), (dval, "val")],
                      evals_result=evals_result,
                      early_stopping_rounds=30,
                      verbose_eval=10)

    val_set_copy = val_set.copy()
    val_set_copy["score"] = model.predict(xgb.DMatrix(val_set_copy[features]))

    def ndcg_at_k(relevance_scores, k):
        relevance_scores = np.array(relevance_scores)
        ideal_relevance = np.sort(relevance_scores)[::-1]  # 降順ソート

        # kに満たない場合、ゼロ埋め
        if len(relevance_scores) < k:
            relevance_scores = np.pad(relevance_scores, (0, k - len(relevance_scores)), constant_values=0)
            ideal_relevance = np.pad(ideal_relevance, (0, k - len(ideal_relevance)), constant_values=0)

        dcg = np.sum(relevance_scores[:k] / np.log2(np.arange(2, k + 2)))
        ideal_dcg = np.sum(ideal_relevance[:k] / np.log2(np.arange(2, k + 2)))

        return dcg / ideal_dcg if ideal_dcg > 0 else 0

    ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()
    ndcg_scores.append(ndcg_val)

[0]	train-ndcg:0.98311	val-ndcg:0.99399
[10]	train-ndcg:0.99166	val-ndcg:0.99686
[20]	train-ndcg:0.99243	val-ndcg:0.99711
[30]	train-ndcg:0.99296	val-ndcg:0.99730
[40]	train-ndcg:0.99346	val-ndcg:0.99750
[50]	train-ndcg:0.99405	val-ndcg:0.99771
[60]	train-ndcg:0.99471	val-ndcg:0.99798
[70]	train-ndcg:0.99499	val-ndcg:0.99809
[80]	train-ndcg:0.99541	val-ndcg:0.99824
[90]	train-ndcg:0.99563	val-ndcg:0.99834
[100]	train-ndcg:0.99583	val-ndcg:0.99842
[110]	train-ndcg:0.99614	val-ndcg:0.99854
[120]	train-ndcg:0.99645	val-ndcg:0.99863
[130]	train-ndcg:0.99659	val-ndcg:0.99870
[140]	train-ndcg:0.99693	val-ndcg:0.99883
[150]	train-ndcg:0.99717	val-ndcg:0.99891
[160]	train-ndcg:0.99740	val-ndcg:0.99901
[170]	train-ndcg:0.99755	val-ndcg:0.99907
[180]	train-ndcg:0.99778	val-ndcg:0.99915
[190]	train-ndcg:0.99809	val-ndcg:0.99923
[200]	train-ndcg:0.99826	val-ndcg:0.99931
[210]	train-ndcg:0.99837	val-ndcg:0.99934
[220]	train-ndcg:0.99851	val-ndcg:0.99940
[230]	train-ndcg:0.99866	val-ndcg:0.99947
[24

/tmp/ipykernel_24774/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


[0]	train-ndcg:0.98320	val-ndcg:0.99425
[10]	train-ndcg:0.99167	val-ndcg:0.99690
[20]	train-ndcg:0.99233	val-ndcg:0.99716
[30]	train-ndcg:0.99295	val-ndcg:0.99739
[40]	train-ndcg:0.99348	val-ndcg:0.99758
[50]	train-ndcg:0.99407	val-ndcg:0.99781
[60]	train-ndcg:0.99469	val-ndcg:0.99809
[70]	train-ndcg:0.99510	val-ndcg:0.99824
[80]	train-ndcg:0.99547	val-ndcg:0.99839
[90]	train-ndcg:0.99577	val-ndcg:0.99850
[100]	train-ndcg:0.99597	val-ndcg:0.99857
[110]	train-ndcg:0.99619	val-ndcg:0.99865
[120]	train-ndcg:0.99649	val-ndcg:0.99872
[130]	train-ndcg:0.99673	val-ndcg:0.99878
[140]	train-ndcg:0.99692	val-ndcg:0.99884
[150]	train-ndcg:0.99734	val-ndcg:0.99899
[160]	train-ndcg:0.99768	val-ndcg:0.99910
[170]	train-ndcg:0.99788	val-ndcg:0.99918
[180]	train-ndcg:0.99801	val-ndcg:0.99923
[190]	train-ndcg:0.99818	val-ndcg:0.99930
[200]	train-ndcg:0.99833	val-ndcg:0.99936
[210]	train-ndcg:0.99844	val-ndcg:0.99939
[220]	train-ndcg:0.99850	val-ndcg:0.99940
[230]	train-ndcg:0.99858	val-ndcg:0.99942
[24

/tmp/ipykernel_24774/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


[0]	train-ndcg:0.98298	val-ndcg:0.99416
[10]	train-ndcg:0.99154	val-ndcg:0.99697
[20]	train-ndcg:0.99186	val-ndcg:0.99704
[30]	train-ndcg:0.99270	val-ndcg:0.99734
[40]	train-ndcg:0.99343	val-ndcg:0.99766
[50]	train-ndcg:0.99407	val-ndcg:0.99791
[60]	train-ndcg:0.99473	val-ndcg:0.99808
[70]	train-ndcg:0.99512	val-ndcg:0.99823
[80]	train-ndcg:0.99540	val-ndcg:0.99833
[90]	train-ndcg:0.99563	val-ndcg:0.99841
[100]	train-ndcg:0.99580	val-ndcg:0.99848
[110]	train-ndcg:0.99610	val-ndcg:0.99860
[120]	train-ndcg:0.99626	val-ndcg:0.99867
[130]	train-ndcg:0.99649	val-ndcg:0.99875
[140]	train-ndcg:0.99689	val-ndcg:0.99887
[150]	train-ndcg:0.99712	val-ndcg:0.99895
[160]	train-ndcg:0.99764	val-ndcg:0.99915
[170]	train-ndcg:0.99798	val-ndcg:0.99928
[180]	train-ndcg:0.99807	val-ndcg:0.99931
[190]	train-ndcg:0.99815	val-ndcg:0.99934
[200]	train-ndcg:0.99829	val-ndcg:0.99940
[210]	train-ndcg:0.99839	val-ndcg:0.99944
[220]	train-ndcg:0.99850	val-ndcg:0.99947
[230]	train-ndcg:0.99864	val-ndcg:0.99950
[24

/tmp/ipykernel_24774/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


[0]	train-ndcg:0.98271	val-ndcg:0.99407
[10]	train-ndcg:0.99192	val-ndcg:0.99716
[20]	train-ndcg:0.99267	val-ndcg:0.99738
[30]	train-ndcg:0.99311	val-ndcg:0.99753
[40]	train-ndcg:0.99371	val-ndcg:0.99771
[50]	train-ndcg:0.99430	val-ndcg:0.99794
[60]	train-ndcg:0.99474	val-ndcg:0.99812
[70]	train-ndcg:0.99523	val-ndcg:0.99829
[80]	train-ndcg:0.99555	val-ndcg:0.99842
[90]	train-ndcg:0.99586	val-ndcg:0.99852
[100]	train-ndcg:0.99611	val-ndcg:0.99861
[110]	train-ndcg:0.99628	val-ndcg:0.99869
[120]	train-ndcg:0.99650	val-ndcg:0.99877
[130]	train-ndcg:0.99666	val-ndcg:0.99882
[140]	train-ndcg:0.99690	val-ndcg:0.99891
[150]	train-ndcg:0.99752	val-ndcg:0.99914
[160]	train-ndcg:0.99791	val-ndcg:0.99927
[170]	train-ndcg:0.99819	val-ndcg:0.99936
[180]	train-ndcg:0.99826	val-ndcg:0.99938
[190]	train-ndcg:0.99844	val-ndcg:0.99944
[200]	train-ndcg:0.99861	val-ndcg:0.99949
[210]	train-ndcg:0.99869	val-ndcg:0.99953
[220]	train-ndcg:0.99877	val-ndcg:0.99956
[230]	train-ndcg:0.99883	val-ndcg:0.99957
[24

/tmp/ipykernel_24774/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


[0]	train-ndcg:0.98333	val-ndcg:0.99428
[10]	train-ndcg:0.99192	val-ndcg:0.99698
[20]	train-ndcg:0.99266	val-ndcg:0.99726
[30]	train-ndcg:0.99305	val-ndcg:0.99744
[40]	train-ndcg:0.99354	val-ndcg:0.99760
[50]	train-ndcg:0.99392	val-ndcg:0.99776
[60]	train-ndcg:0.99454	val-ndcg:0.99799
[70]	train-ndcg:0.99491	val-ndcg:0.99816
[80]	train-ndcg:0.99528	val-ndcg:0.99828
[90]	train-ndcg:0.99553	val-ndcg:0.99840
[100]	train-ndcg:0.99586	val-ndcg:0.99853
[110]	train-ndcg:0.99611	val-ndcg:0.99862
[120]	train-ndcg:0.99643	val-ndcg:0.99877
[130]	train-ndcg:0.99659	val-ndcg:0.99884
[140]	train-ndcg:0.99686	val-ndcg:0.99893
[150]	train-ndcg:0.99704	val-ndcg:0.99898
[160]	train-ndcg:0.99741	val-ndcg:0.99909
[170]	train-ndcg:0.99776	val-ndcg:0.99918
[180]	train-ndcg:0.99802	val-ndcg:0.99929
[190]	train-ndcg:0.99840	val-ndcg:0.99945
[200]	train-ndcg:0.99856	val-ndcg:0.99950
[210]	train-ndcg:0.99871	val-ndcg:0.99955
[220]	train-ndcg:0.99879	val-ndcg:0.99958
[230]	train-ndcg:0.99888	val-ndcg:0.99960
[24

/tmp/ipykernel_24774/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


In [13]:
print(f"Average nDCG@22 from CV: {np.mean(ndcg_scores):.4f}")

Average nDCG@22 from CV: 0.9749


In [14]:
# テストデータの処理
candidate_products = train_data.groupby("product_id")["user_id"].count().reset_index()
candidate_products = candidate_products.sort_values("user_id", ascending=False).head(22)  # 上位22商品

In [15]:
# すべてのユーザーに候補商品を紐づける
test_data = test_data.assign(key=1).merge(candidate_products.assign(key=1), on="key").drop("key", axis=1)
test_data = test_data.rename(columns={'user_id_x': 'user_id'}).drop(columns=['user_id_y'])

In [16]:
# 評価データの前処理（学習データと同じ特徴量処理を適用）
test_data = test_data.merge(user_features, on="user_id", how="left")
test_data = test_data.merge(product_features, on="product_id", how="left")
test_data = test_data.merge(interaction_features, on=["user_id", "product_id"], how="left")
test_data.fillna(0, inplace=True)

In [17]:
# 予測用データ
X_test = test_data[features]
dtest = xgb.DMatrix(X_test)

In [18]:
# 予測スコアの計算
test_data["score"] = model.predict(dtest)

In [19]:
# 各ユーザーごとにランキング付け（スコアが高い順）
test_data["rank"] = test_data.groupby("user_id")["score"].rank(ascending=False, method="first")

In [20]:
# テストデータの nDCG 計算
ground_truth_test = defaultdict(dict)
for _, row in train_data.iterrows():
    ground_truth_test[row["user_id"]][row["product_id"]] = row["relevance"]

def evaluate_ndcg(data, ground_truth, k=22):
    ndcg_scores = []
    for user_id, group in data.groupby("user_id"):
        predicted_items = group.sort_values("score", ascending=False)["product_id"].tolist()
        relevance_scores = [ground_truth[user_id].get(item, 0) for item in predicted_items]
        ndcg_scores.append(ndcg_at_k(relevance_scores, k))
    return np.mean(ndcg_scores) if ndcg_scores else 0

ndcg_test = evaluate_ndcg(test_data, ground_truth_test, k=22)
print(f"Test nDCG@22: {ndcg_test:.4f}")

Test nDCG@22: 0.0342


In [21]:
# 提出用データの作成（上位 22 件のみ）
submission = test_data[test_data["rank"] <= 22][["user_id", "product_id", "rank"]]
submission['rank'] = submission['rank'].astype(int)

In [22]:
# 保存
submission.to_csv('./predict_file/predictions_C.tsv', sep="\t", index=False)